## Analysis of Hatexplain and SST Datasets with New Metrics

In this notebook, we evaluate two datasets: **Hatexplain** and **SST**. We use the new metrics of **confidence** and **rationale consistency** to enhance our analysis.

### Procedure

1. **Confidence**:
   - Train a linear regression model using the first 49 entries of the dataset.
   - Evaluate the model with the 50th entry to measure the confidence metric.

2. **Rationale Consistency**:
   - Measure consistency using Spearman's correlation among the first 50 entries of the dataset.
   - This metric helps determine the consistency of rational explanations across different entries.

### Objectives

- **Confidence**: Establish the reliability of the model's predictions.
- **Rationale Consistency**: Assess the coherence of rationales provided by the model across dataset entries.

These analyses will help us better understand the performance and reliability of models in text classification contexts.


In [3]:
# rationale test -dataset


from transformers import AutoModelForSequenceClassification, AutoTokenizer
from ferret.benchmark import Benchmark
from ferret.evaluators.evaluation import  ExplanationEvaluation
import torch
from tqdm.autonotebook import tqdm



name = "cardiffnlp/twitter-xlm-roberta-base-sentiment"
model = AutoModelForSequenceClassification.from_pretrained(name)
tokenizer = AutoTokenizer.from_pretrained(name)



# Initialize the model with random weights
model_rand = AutoModelForSequenceClassification.from_pretrained(name)

        # Re-initialize the weights to random values
def initialize_weights(module):
      if isinstance(module, (torch.nn.Linear, torch.nn.Embedding)):
                module.reset_parameters()
      elif isinstance(module, torch.nn.LayerNorm):
                module.bias.data.zero_()
                module.weight.data.fill_(1.0)

model_rand.apply(initialize_weights)



bench = Benchmark(model, tokenizer)
bench_rand = Benchmark(model_rand, tokenizer)

target = 1




`resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.


In [2]:
dataset = bench.load_dataset("hatexplain")

list_input=[]
for i in range(50):  
        
        text = dataset[i]["text"].replace("<user>", "")

        explanations      = bench.explain(text, target=target)
        explanations_rand = bench_rand.explain(text, target=target)

        explanations_randAll_scores = [exp.all_scores for exp in explanations_rand]

        for exp, exp2_all_scores in zip(explanations, explanations_randAll_scores):
             exp.all_scores_rand = exp2_all_scores

        list_input.append(explanations)



evaluations = bench.evaluate_explanations(list_input, target=target,  name = name)

# check multiple input
if isinstance(evaluations[0].explanation, list):
      
    for evaluation in evaluations:
        evaluation.explanation = evaluation.explanation[0]
   


bench.show_evaluation_table(evaluations)

Explanation eval:   0%|          | 0/50 [00:00<?, ?it/s] `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Explanation eval:   2%|▏         | 1/50 [00:24<20:16, 24.82s/it]`resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Explanation eval:   4%|▍         | 2/50 [00:50<20:12, 25.25s/it]`resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
Asking to trun

,ci,rat_cons
Explainer,,
Partition SHAP,0.02,0.08
LIME,0.02,0.01
Gradient,0.19,-0.25
Gradient (x Input),0.23,-0.05
Integrated Gradient,0.21,-0.02
Integrated Gradient (x Input),0.17,0.16


In [4]:
dataset = bench.load_dataset("sst")

list_input=[]
for i in range(50):  
        
        
        text = dataset[i]["text"]

        explanations      = bench.explain(text, target=target)
        explanations_rand = bench_rand.explain(text, target=target)

        explanations_randAll_scores = [exp.all_scores for exp in explanations_rand]

        for exp, exp2_all_scores in zip(explanations, explanations_randAll_scores):
             exp.all_scores_rand = exp2_all_scores

        list_input.append(explanations)



evaluations = bench.evaluate_explanations(list_input, target=target,  name = name)

# check multiple input
if isinstance(evaluations[0].explanation, list):
      
    for evaluation in evaluations:
        evaluation.explanation = evaluation.explanation[0]
   


bench.show_evaluation_table(evaluations)

Explanation eval:   0%|          | 0/50 [00:00<?, ?it/s] `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Explanation eval:   2%|▏         | 1/50 [00:35<28:56, 35.45s/it]`resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Explanation eval:   4%|▍         | 2/50 [01:07<26:54, 33.63s/it]`resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
Asking to trun

,ci,rat_cons
Explainer,,
Partition SHAP,0.10,0.01
LIME,0.04,0.21
Gradient,0.05,-0.15
Gradient (x Input),0.03,-0.03
Integrated Gradient,0.03,0.10
Integrated Gradient (x Input),0.03,0.11
